In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
#https://docs.langchain.com/oss/python/integrations/splitters/index


from unstructured.partition.auto import partition
from pathlib import Path

ModuleNotFoundError: No module named 'langchain_text_splitters'

In [ ]:
path = Path(r"C:\Users\Admin\Desktop\ip\How To RAG\docs")

documents = []
id = 0

for docs in path.iterdir():
#print (docs)
    elements = partition (str(docs))
    #print("\n\n".join ([str(el) for el in elements]))  
    for el in elements:
        documents.append ({
                "id": f"{docs.name}_{id}",
                "text": el.text,
                "metadata": {
                    "source": docs.name,
                    "type": el.category,
                    "file_type": el.metadata.to_dict()["filetype"]
                    #"language": el.metadata.to_dict()["languages"]
            }
        })

        id += 1
#print (documents)

In [ ]:
display (documents)

#display (documents[1]["text"])
#print (documents[0]["metadata"])

#for item in documents:
    #print (item["metadata"]["source"])

In [ ]:
docs = [
    Document (
        page_content = item["text"], 
        metadata = {
            "id": item["id"],
            "source": item["metadata"]["source"],
            "type": item["metadata"]["type"],
            "file_type": item["metadata"]["file_type"]
        }
    )
    for item in documents
]

In [ ]:
display (docs)

In [ ]:
#RecursiveCharacterTextSplitter
#CharacterTextSplitter
#TokenTextSplitter
#SentenceTransformersTokenTextSplitter
#MarkdownHeaderTextSplitter
#HTMLHeaderTextSplitter
#PythonCodeTextSplitter
#RecursiveJsonSplitter

chunking = RecursiveCharacterTextSplitter (
    chunk_size = 800, #800 bom ponto de partida
    chunk_overlap = 100 #10 a 20% do chunk_size
) 

chunks = chunking.split_documents (docs)

In [ ]:
display (chunks)

<h3> Embedding </h3>

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings #Para dar Load de um modelo de embeddings
from langchain_community.vectorstores import FAISS #https://docs.langchain.com/oss/python/integrations/vectorstores

In [ ]:
embeddings = HuggingFaceEmbeddings (model_name = r"C:\Users\Admin\Desktop\models\Embedding Models") #https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
db = FAISS.from_documents(chunks, embeddings) #direto para vector database

In [ ]:
#display (db.docstore._dict)
#display (db.index.ntotal) #Número de Chunks
#print (db.index_to_docstore_id)
#db.save_local("x")

<h3> Embedding da Pergunta </h3>
<h3> Similiariedade </h3>

In [ ]:
query = "Que país apoiou a invasão italiana da Etiópia quando outras grandes potências europeias não o fizeram?"
embedding_vector = embeddings.embed_query(query)

print (embedding_vector)

In [ ]:
busca = db.similarity_search ("Que país apoiou a invasão italiana da Etiópia quando outras grandes potências europeias não o fizeram?", k = 3)
busca2 = db.similarity_search_with_relevance_scores ("Que país apoiou a invasão italiana da Etiópia quando outras grandes potências europeias não o fizeram?", k = 3)
busca3 = db.similarity_search_by_vector (embedding_vector, k = 3)
busca4 = db.similarity_search_with_score_by_vector (embedding_vector, k = 3)


In [ ]:
for i in busca2:
    print (i[1])
    #print (i[0])

for i in busca4:
    print (i[1])
    #print (i[0])

<h3> Retriever </h3>

In [ ]:
retriever = db.as_retriever ( #Converte db em Retriever
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 10
    }
)

retrrrr = retriever.get_relevant_documents("Qual era a designação da REFER após a alteração de 2008?")

In [ ]:
#print (retrrrr[i].metadata["source"] for i in range (2))
for i in retrrrr:
    print (i)



In [ ]:
import pandas as pd 

df = pd.read_json (r"C:\Users\Admin\Desktop\ip\val_dataset.json")

for _, item in df.iterrows():
    print (item["doc"])

Recall = (número de documentos relevantes recuperados) / (número total de documentos relevantes existentes)

In [ ]:
def recall (retriever, df):
    scores = []

    for i, item in df.iterrows():

        pergunta = item["question"]
        #print (pergunta)
        docs = (item["doc"]) if isinstance(item["doc"], list) else {item["doc"]}

        #print (docs)

        retr = retriever.get_relevant_documents (pergunta)

        retr_source = set(d.metadata.get("source") for d in retr)
        #print (len(retr_source))

        recall = len(docs & retr_source) / len(docs)

        scores.append(recall)

        #print (f"Pergunta: {pergunta}")
        #print (f"Recall@Score: {recall}")
        
    return (sum(scores) / len(scores))


In [ ]:
recall (retriever, df)

<h3> Índice BM25 </h3>

In [ ]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents (docs)

In [ ]:
query = "Qual era a designação da REFER após a alteração de 2008?"
scores = bm25_retriever.vectorizer.get_scores (query.lower().split())

# Pair each document with its score
results_with_scores = list(zip(docs, scores))

for doc, score in results_with_scores:
    print(f"Score: {score:.4f} | Content: {doc.page_content[:200]}") 

In [ ]:
hum = bm25_retriever.invoke ("Qual era a designação da REFER após a alteração de 2008?")
display (hum)



In [ ]:
resultados = bm25_retriever.vectorizer.get_scores("Qual era a designação da REFER após 2008?")

print (resultados)

<h2> RFF </h2>

In [ ]:
query = "Que país apoiou a invasão italiana da Etiópia quando outras grandes potências europeias não o fizeram?"

bm25_results = bm25_retriever.invoke (query)
vector_results = retriever.invoke (query)

#print (bm25_results)
#print (vector_results)

In [ ]:
def rrf(rankings, k = 60):
    scores = {}

    for ranking in rankings:
        for rank, doc in enumerate(ranking):
            doc_id = doc.page_content

            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)

    return sorted(scores.items(), key = lambda x: x[1], reverse = True)

In [ ]:
final = rrf([bm25_results, vector_results])

In [ ]:
display (final[:200])

In [372]:
x = []

for y in final:
    x.append (y[0])
    #print (y[0])

<h3> Cross Encoder </h3>

In [ ]:
#from transformers import AutoTokenizer, AutoModelForSequenceClassification
#import torch

#model = AutoModelForSequenceClassification.from_pretrained('cross-encoder/ms-marco-MiniLM-L6-v2')
#tokenizer = AutoTokenizer.from_pretrained('cross-encoder/ms-marco-MiniLM-L6-v2')

#model.save_pretrained (r"C:\Users\Admin\Desktop\models\Cross Encoder")
#tokenizer.save_pretrained (r"C:\Users\Admin\Desktop\models\Cross Encoder")

In [375]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model = AutoModelForSequenceClassification.from_pretrained(r'C:\Users\Admin\Desktop\models\Cross Encoder')
tokenizer = AutoTokenizer.from_pretrained(r"C:\Users\Admin\Desktop\models\Cross Encoder")




Loading weights: 100%|██████████| 105/105 [00:00<00:00, 5090.76it/s]


In [382]:
inputs = tokenizer(
    [query] * len(docs),
    [d.page_content for d in docs],
    padding=True,
    truncation=True,
    return_tensors="pt"
)

with torch.no_grad():
    scores = model(**inputs).logits.squeeze(-1)


reranked = sorted(
    zip(docs, scores.tolist()),
    key=lambda x: x[1],
    reverse=True
)


In [383]:
top_docs = [doc for doc, score in reranked[:5]]

In [384]:
for i in top_docs:
    print (i)

page_content='Segunda Guerra Mundial foi um conflito militar global que durou de 1939 a 1945, envolvendo a maioria das nações do mundo — incluindo todas as grandes potências — organizadas em duas alianças militares opostas: os Aliados e o Eixo. Foi a guerra mais abrangente da história, com mais de 100 milhões de militares mobilizados. Em estado de "guerra total", os principais envolvidos dedicaram toda sua capacidade econômica, industrial e científica a serviço dos esforços de guerra, deixando de lado a distinção entre recursos civis e militares. Marcado por um número significante de ataques contra civis, incluindo o Holocausto e a única vez em que armas nucleares foram utilizadas em combate, foi o conflito mais letal da história da humanidade, que resultou na morte de 50-70 milhões de pessoas. Geralmente considera-se o ponto inicial da guerra como sendo a invasão da Polônia pela Alemanha Nazista em 1 de setembro de 1939 e subsequentes declarações de guerra contra a Alemanha pela Franç